# Tokenizer Comparison: BPE vs WordPiece vs SentencePiece

> Compare different tokenization strategies used by major LLMs

| Model | Tokenizer | Vocab Size |
|-------|-----------|------------|
| GPT-2/3/4 | BPE | 50,257 |
| BERT | WordPiece | 30,522 |
| T5 | SentencePiece | 32,100 |
| LLaMA | BPE | 32,000 |
| Mistral | BPE | 32,000 |

In [ ]:
import re
import json
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np

## 1. BPE (Byte-Pair Encoding)

Used by: GPT-2, GPT-3, GPT-4, LLaMA, Mistral

**Algorithm:**
1. Start with character vocabulary
2. Iteratively merge most frequent pairs
3. Stop when vocab size reached

In [ ]:
class BPE:
    """Byte-Pair Encoding from scratch"""
    
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = []
    
    def get_stats(self, vocab):
        """Count frequency of adjacent pairs"""
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs
    
    def merge_vocab(self, pair, vocab):
        """Merge all occurrences of pair"""
        bigram = re.escape(' '.join(pair))
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        new_vocab = {}
        for word in vocab:
            new_word = pattern.sub(''.join(pair), word)
            new_vocab[new_word] = vocab[word]
        return new_vocab
    
    def train(self, text):
        """Train BPE on text"""
        # Pre-tokenize into words
        words = text.split()
        
        # Build initial vocabulary (characters + </w>)
        vocab = {}
        for word in words:
            word_chars = ' '.join(list(word)) + ' </w>'
            vocab[word_chars] = vocab.get(word_chars, 0) + 1
        
        # Initial tokens
        self.vocab = set()
        for word in vocab:
            self.vocab.update(word.split())
        
        num_merges = self.vocab_size - len(self.vocab)
        
        for i in range(num_merges):
            pairs = self.get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self.merge_vocab(best, vocab)
            self.merges.append(best)
            self.vocab.add(''.join(best))
        
        return self
    
    def encode(self, text):
        """Encode text to token IDs"""
        tokens = []
        for word in text.split():
            word_tokens = list(word) + ['</w>']
            for merge in self.merges:
                new_tokens = []
                i = 0
                while i < len(word_tokens):
                    if i < len(word_tokens) - 1 and (word_tokens[i], word_tokens[i+1]) == merge:
                        new_tokens.append(''.join(merge))
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            tokens.extend(word_tokens)
        return tokens
    
    def decode(self, tokens):
        """Decode tokens back to text"""
        text = ''.join(tokens)
        text = text.replace('</w>', ' ')
        return text.strip()

# Test BPE
sample_text = """
low lower lowest new newer newest wide wider widest
fast faster fastest slow slower slowest
"""

bpe = BPE(vocab_size=50)
bpe.train(sample_text)

print(f"BPE Merges: {len(bpe.merges)}")
print(f"BPE Vocab size: {len(bpe.vocab)}")
print(f"\nFirst 10 merges: {bpe.merges[:10]}")

## 2. WordPiece

Used by: BERT, DistilBERT, RoBERTa

**Difference from BPE:**
- BPE: merges by **frequency**
- WordPiece: merges by **likelihood** (language model score)
- WordPiece uses `##` prefix for subword continuation

In [ ]:
class WordPiece:
    """WordPiece tokenization (simplified)"""
    
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {'[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'}
        self.word_counts = Counter()
    
    def train(self, text):
        """Train WordPiece (simplified version)"""
        words = text.lower().split()
        self.word_counts = Counter(words)
        
        # Start with character vocabulary
        char_vocab = set()
        for word in self.word_counts:
            char_vocab.update(word)
        
        self.vocab.update(char_vocab)
        
        # Build subword vocabulary (simplified greedy approach)
        subword_scores = {}
        for word, count in self.word_counts.most_common():
            for i in range(len(word)):
                for j in range(i + 2, min(i + 8, len(word) + 1)):
                    subword = word[i:j]
                    if subword not in self.vocab:
                        subword_scores[subword] = subword_scores.get(subword, 0) + count
        
        # Add top subwords by frequency
        for subword, _ in sorted(subword_scores.items(), key=lambda x: -x[1]):
            if len(self.vocab) >= self.vocab_size:
                break
            self.vocab.add(subword)
        
        return self
    
    def encode_word(self, word):
        """Greedy longest-match encoding"""
        tokens = []
        i = 0
        is_first = True
        
        while i < len(word):
            # Try longest match
            for j in range(min(i + 10, len(word)), i, -1):
                subword = word[i:j]
                if subword in self.vocab:
                    if not is_first and len(tokens) > 0:
                        tokens.append('##' + subword)
                    else:
                        tokens.append(subword)
                    is_first = False
                    i = j
                    break
            else:
                # Unknown character
                tokens.append('[UNK]')
                i += 1
        
        return tokens
    
    def encode(self, text):
        words = text.lower().split()
        tokens = ['[CLS]']
        for word in words:
            tokens.extend(self.encode_word(word))
        tokens.append('[SEP]')
        return tokens
    
    def decode(self, tokens):
        text = ' '.join(tokens)
        text = text.replace(' ##', '').replace('[CLS]', '').replace('[SEP]', '')
        return text.strip()

# Test WordPiece
wp = WordPiece(vocab_size=50)
wp.train(sample_text)

print(f"WordPiece Vocab size: {len(wp.vocab)}")
print(f"\nSample tokens: {wp.encode('lowest newer')}")

## 3. SentencePiece

Used by: T5, XLNet, ALBERT

**Key Features:**
- Language-agnostic (works on raw text, no pre-tokenization)
- Uses BPE or Unigram algorithm
- Adds `▁` (U+2581) prefix for word boundaries
- Can encode any Unicode text

In [ ]:
class SentencePiece:
    """Simplified SentencePiece (BPE-based)"""
    
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = set()
        self.merges = []
    
    def train(self, text):
        """Train SentencePiece-style BPE"""
        # Add word boundary marker ▁
        words = text.split()
        vocab = {}
        
        for word in words:
            # Prefix with ▁
            word_marked = '▁' + ' '.join(list(word))
            vocab[word_marked] = vocab.get(word_marked, 0) + 1
        
        # Build initial vocab
        self.vocab = {'▁'}
        for word in vocab:
            self.vocab.update(word.split())
        
        # BPE merging
        num_merges = self.vocab_size - len(self.vocab)
        
        for _ in range(num_merges):
            pairs = defaultdict(int)
            for word, freq in vocab.items():
                symbols = word.split()
                for i in range(len(symbols) - 1):
                    pairs[(symbols[i], symbols[i+1])] += freq
            
            if not pairs:
                break
            
            best = max(pairs, key=pairs.get)
            bigram = re.escape(' '.join(best))
            pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
            
            new_vocab = {}
            for word in vocab:
                new_word = pattern.sub(''.join(best), word)
                new_vocab[new_word] = vocab[word]
            vocab = new_vocab
            
            self.merges.append(best)
            self.vocab.add(''.join(best))
        
        return self
    
    def encode(self, text):
        tokens = []
        for word in text.split():
            word_tokens = ['▁'] + list(word)
            for merge in self.merges:
                new_tokens = []
                i = 0
                while i < len(word_tokens):
                    if i < len(word_tokens) - 1 and (word_tokens[i], word_tokens[i+1]) == merge:
                        new_tokens.append(''.join(merge))
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            tokens.extend(word_tokens)
        return tokens
    
    def decode(self, tokens):
        text = ''.join(tokens)
        text = text.replace('▁', ' ').strip()
        return text

# Test SentencePiece
sp = SentencePiece(vocab_size=50)
sp.train(sample_text)

print(f"SentencePiece Vocab size: {len(sp.vocab)}")
print(f"\nSample tokens: {sp.encode('lowest newer')}")

## 4. Comparison Analysis

In [ ]:
# Test sentence
test_sentences = [
    "lowest newer fastest",
    "unhappiness running",
    "hello world 123",
    "AI is amazing",
]

results = []
for sentence in test_sentences:
    bpe_tokens = bpe.encode(sentence)
    wp_tokens = wp.encode(sentence)
    sp_tokens = sp.encode(sentence)
    
    results.append({
        'sentence': sentence,
        'bpe_len': len(bpe_tokens),
        'wp_len': len(wp_tokens),
        'sp_len': len(sp_tokens),
        'bpe': bpe_tokens,
        'wp': wp_tokens,
        'sp': sp_tokens,
    })

# Display results
for r in results:
    print(f"\n📝 Sentence: '{r['sentence']}'")
    print(f"   BPE ({r['bpe_len']} tokens): {r['bpe']}")
    print(f"   WP  ({r['wp_len']} tokens): {r['wp']}")
    print(f"   SP  ({r['sp_len']} tokens): {r['sp']}")

In [ ]:
# Visualization: Token length comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of token counts
sentences_short = [r['sentence'][:15] + '...' if len(r['sentence']) > 15 else r['sentence'] for r in results]
x = np.arange(len(sentences_short))
width = 0.25

axes[0].bar(x - width, [r['bpe_len'] for r in results], width, label='BPE', color='#FF6B6B')
axes[0].bar(x, [r['wp_len'] for r in results], width, label='WordPiece', color='#4ECDC4')
axes[0].bar(x + width, [r['sp_len'] for r in results], width, label='SentencePiece', color='#45B7D1')

axes[0].set_xlabel('Sentences')
axes[0].set_ylabel('Number of Tokens')
axes[0].set_title('Token Count Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(sentences_short, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Vocabulary size comparison
methods = ['BPE', 'WordPiece', 'SentencePiece']
vocab_sizes = [len(bpe.vocab), len(wp.vocab), len(sp.vocab)]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[1].bar(methods, vocab_sizes, color=colors, edgecolor='black')
axes[1].set_ylabel('Vocabulary Size')
axes[1].set_title('Vocabulary Size Comparison')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(vocab_sizes):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Key Differences Summary

| Feature | BPE | WordPiece | SentencePiece |
|---------|-----|-----------|---------------|
| **Merge Criterion** | Frequency | Likelihood | Frequency or Unigram |
| **Pre-tokenization** | Required (spaces) | Required (spaces) | **Not required** |
| **Language** | English-focused | English-focused | **Language-agnostic** |
| **Special Tokens** | None | [CLS], [SEP], [MASK] | User-defined |
| **Subword Marker** | None | `##` | `▁` |
| **Unknown Words** | Character fallback | [UNK] | Character fallback |
| **Used By** | GPT, LLaMA, Mistral | BERT, RoBERTa | T5, XLNet |

## 🎯 Exercises

1. **Vocab Size Impact**: Train all three with vocab_size=200 vs 500. Compare tokenization quality.
2. **Multilingual Test**: Test with non-English text (Bangla, Arabic, Chinese).
3. **Unknown Words**: How does each handle 'supercalifragilisticexpialidocious'?
4. **Merge Analysis**: Visualize the first 20 merges for BPE. What patterns emerge?
5. **Efficiency**: Compare encoding speed for a 1000-word paragraph.
6. **Hugging Face**: Use `AutoTokenizer` to compare with real implementations.